In [0]:
print("Healthcare Medallion Project")
print("Databricks is working!")

Healthcare Medallion Project
Databricks is working!


In [0]:
df = spark.range(10)

display(df)


id
0
1
2
3
4
5
6
7
8
9


In [0]:
display(dbutils.fs.ls("/Volumes/healthcare/default/healthcare_landing/"))

path,name,size,modificationTime
dbfs:/Volumes/healthcare/default/healthcare_landing/appointments.csv,appointments.csv,10907,1786368629000
dbfs:/Volumes/healthcare/default/healthcare_landing/billing.csv,billing.csv,10018,1786368629000
dbfs:/Volumes/healthcare/default/healthcare_landing/doctors.csv,doctors.csv,962,1786368629000
dbfs:/Volumes/healthcare/default/healthcare_landing/patients.csv,patients.csv,5626,1786368629000
dbfs:/Volumes/healthcare/default/healthcare_landing/treatments.csv,treatments.csv,11072,1786368629000


In [0]:
patients_path = "/Volumes/healthcare/default/healthcare_landing/patients.csv"

patients_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(patients_path)
)

display(patients_df)

patient_id,first_name,last_name,gender,date_of_birth,contact_number,address,registration_date,insurance_provider,insurance_number,email
P001,David,Williams,F,1955-06-04,6939585183,789 Pine Rd,2022-06-23,WellnessCorp,INS840674,david.williams@mail.com
P002,Emily,Smith,F,1984-10-12,8228188767,321 Maple Dr,2022-01-15,PulseSecure,INS354079,emily.smith@mail.com
P003,Laura,Jones,M,1977-08-21,8397029847,321 Maple Dr,2022-02-07,PulseSecure,INS650929,laura.jones@mail.com
P004,Michael,Johnson,F,1981-02-20,9019443432,123 Elm St,2021-03-02,HealthIndia,INS789944,michael.johnson@mail.com
P005,David,Wilson,M,1960-06-23,7734463155,123 Elm St,2021-09-29,MedCare Plus,INS788105,david.wilson@mail.com
P006,Linda,Jones,M,1963-06-16,7561777264,321 Maple Dr,2022-10-02,HealthIndia,INS613758,linda.jones@mail.com
P007,Alex,Johnson,F,1989-06-08,6278710077,789 Pine Rd,2021-12-25,MedCare Plus,INS465890,alex.johnson@mail.com
P008,David,Davis,F,1976-07-05,7090558393,456 Oak Ave,2021-05-25,WellnessCorp,INS545101,david.davis@mail.com
P009,Laura,Davis,M,1971-12-11,7060324619,321 Maple Dr,2022-09-18,PulseSecure,INS136631,laura.davis@mail.com
P010,Michael,Taylor,M,2001-10-13,7081396733,123 Elm St,2022-08-24,WellnessCorp,INS866577,michael.taylor@mail.com


In [0]:
appointments_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv("/Volumes/healthcare/default/healthcare_landing/appointments.csv")
)

billing_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv("/Volumes/healthcare/default/healthcare_landing/billing.csv")
)

doctors_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv("/Volumes/healthcare/default/healthcare_landing/doctors.csv")
)

treatments_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv("/Volumes/healthcare/default/healthcare_landing/treatments.csv")
)

print("All five datasets loaded successfully.")

All five datasets loaded successfully.


In [0]:
datasets = {
    "Patients": patients_df,
    "Appointments": appointments_df,
    "Billing": billing_df,
    "Doctors": doctors_df,
    "Treatments": treatments_df
}

for name, df in datasets.items():
    print(f"\n{'=' * 50}")
    print(name)
    print(f"{'=' * 50}")
    print("Rows:", df.count())
    print("Columns:", len(df.columns))
    print("Column names:", df.columns)


Patients
Rows: 50
Columns: 11
Column names: ['patient_id', 'first_name', 'last_name', 'gender', 'date_of_birth', 'contact_number', 'address', 'registration_date', 'insurance_provider', 'insurance_number', 'email']

Appointments
Rows: 200
Columns: 7
Column names: ['appointment_id', 'patient_id', 'doctor_id', 'appointment_date', 'appointment_time', 'reason_for_visit', 'status']

Billing
Rows: 200
Columns: 7
Column names: ['bill_id', 'patient_id', 'treatment_id', 'bill_date', 'amount', 'payment_method', 'payment_status']

Doctors
Rows: 10
Columns: 8
Column names: ['doctor_id', 'first_name', 'last_name', 'specialization', 'phone_number', 'years_experience', 'hospital_branch', 'email']

Treatments
Rows: 200
Columns: 6
Column names: ['treatment_id', 'appointment_id', 'treatment_type', 'description', 'cost', 'treatment_date']


In [0]:
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    BooleanType,
    IntegerType
)

metadata_schema = StructType([
    StructField("source_id", StringType(), False),
    StructField("source_name", StringType(), False),
    StructField("file_path", StringType(), False),
    StructField("file_format", StringType(), False),
    StructField("target_table", StringType(), False),
    StructField("primary_key", StringType(), False),
    StructField("load_type", StringType(), False),
    StructField("active_flag", BooleanType(), False),
    StructField("data_classification", StringType(), False),
    StructField("description", StringType(), True)
])

metadata_data = [
    (
        "SRC001",
        "patients",
        "/Volumes/healthcare/default/healthcare_landing/patients.csv",
        "csv",
        "healthcare.default.bronze_patients",
        "patient_id",
        "FULL",
        True,
        "PII",
        "Patient master data"
    ),
    (
        "SRC002",
        "appointments",
        "/Volumes/healthcare/default/healthcare_landing/appointments.csv",
        "csv",
        "healthcare.default.bronze_appointments",
        "appointment_id",
        "FULL",
        True,
        "PHI",
        "Patient appointment records"
    ),
    (
        "SRC003",
        "billing",
        "/Volumes/healthcare/default/healthcare_landing/billing.csv",
        "csv",
        "healthcare.default.bronze_billing",
        "bill_id",
        "FULL",
        True,
        "PHI",
        "Hospital billing records"
    ),
    (
        "SRC004",
        "doctors",
        "/Volumes/healthcare/default/healthcare_landing/doctors.csv",
        "csv",
        "healthcare.default.bronze_doctors",
        "doctor_id",
        "FULL",
        True,
        "PII",
        "Doctor master data"
    ),
    (
        "SRC005",
        "treatments",
        "/Volumes/healthcare/default/healthcare_landing/treatments.csv",
        "csv",
        "healthcare.default.bronze_treatments",
        "treatment_id",
        "FULL",
        True,
        "PHI",
        "Patient treatment records"
    )
]

metadata_df = spark.createDataFrame(
    metadata_data,
    schema=metadata_schema
)

display(metadata_df)

source_id,source_name,file_path,file_format,target_table,primary_key,load_type,active_flag,data_classification,description
SRC001,patients,/Volumes/healthcare/default/healthcare_landing/patients.csv,csv,healthcare.default.bronze_patients,patient_id,FULL,true,PII,Patient master data
SRC002,appointments,/Volumes/healthcare/default/healthcare_landing/appointments.csv,csv,healthcare.default.bronze_appointments,appointment_id,FULL,true,PHI,Patient appointment records
SRC003,billing,/Volumes/healthcare/default/healthcare_landing/billing.csv,csv,healthcare.default.bronze_billing,bill_id,FULL,true,PHI,Hospital billing records
SRC004,doctors,/Volumes/healthcare/default/healthcare_landing/doctors.csv,csv,healthcare.default.bronze_doctors,doctor_id,FULL,true,PII,Doctor master data
SRC005,treatments,/Volumes/healthcare/default/healthcare_landing/treatments.csv,csv,healthcare.default.bronze_treatments,treatment_id,FULL,true,PHI,Patient treatment records


In [0]:
metadata_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("healthcare.default.metadata_config")

In [0]:
display(
    spark.table("healthcare.default.metadata_config")
)

source_id,source_name,file_path,file_format,target_table,primary_key,load_type,active_flag,data_classification,description
SRC001,patients,/Volumes/healthcare/default/healthcare_landing/patients.csv,csv,healthcare.default.bronze_patients,patient_id,FULL,true,PII,Patient master data
SRC002,appointments,/Volumes/healthcare/default/healthcare_landing/appointments.csv,csv,healthcare.default.bronze_appointments,appointment_id,FULL,true,PHI,Patient appointment records
SRC003,billing,/Volumes/healthcare/default/healthcare_landing/billing.csv,csv,healthcare.default.bronze_billing,bill_id,FULL,true,PHI,Hospital billing records
SRC004,doctors,/Volumes/healthcare/default/healthcare_landing/doctors.csv,csv,healthcare.default.bronze_doctors,doctor_id,FULL,true,PII,Doctor master data
SRC005,treatments,/Volumes/healthcare/default/healthcare_landing/treatments.csv,csv,healthcare.default.bronze_treatments,treatment_id,FULL,true,PHI,Patient treatment records


In [0]:
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType,
    LongType,
    DoubleType,
    BooleanType,
    TimestampType
)

audit_schema = StructType([
    StructField("audit_id", StringType(), False),
    StructField("batch_id", StringType(), False),
    StructField("source_id", StringType(), True),
    StructField("source_name", StringType(), True),
    StructField("layer", StringType(), False),
    StructField("pipeline_start_time", TimestampType(), True),
    StructField("pipeline_end_time", TimestampType(), True),
    StructField("rows_read", LongType(), True),
    StructField("rows_written", LongType(), True),
    StructField("rows_rejected", LongType(), True),
    StructField("rows_quarantined", LongType(), True),
    StructField("status", StringType(), False),
    StructField("error_message", StringType(), True),
    StructField("pipeline_duration_secs", DoubleType(), True),
    StructField("notebook_name", StringType(), True),
    StructField("environment", StringType(), True),
    StructField("dq_score_avg", DoubleType(), True),
    StructField("sla_met", BooleanType(), True),
    StructField("retry_attempt", IntegerType(), True),
    StructField("data_classification", StringType(), True),
    StructField("downstream_notified", BooleanType(), True)
])

empty_audit_df = spark.createDataFrame([], audit_schema)

empty_audit_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("healthcare.default.audit_log")

print("Audit log table created successfully.")

Audit log table created successfully.


In [0]:
display(
    spark.table("healthcare.default.audit_log")
)

audit_id,batch_id,source_id,source_name,layer,pipeline_start_time,pipeline_end_time,rows_read,rows_written,rows_rejected,rows_quarantined,status,error_message,pipeline_duration_secs,notebook_name,environment,dq_score_avg,sla_met,retry_attempt,data_classification,downstream_notified
